# PCB Router v5 — fixed budget, 16 directions

Branch **`round-robin-v2`** of [`adikeshn/pcb-router-world`](https://github.com/adikeshn/pcb-router-world).

See `docs/reward_spec.md` and `docs/environment_spec.md` for the full design.

Run top to bottom: setup → W&B → hyperparameters → **board preview** →
**acceptance checks** → train → results.


## 1 · Setup

Hard-syncs the branch every run and verifies the checked-out code matches this
notebook. Pulls from GitHub — local edits must be pushed to the branch first.


In [ ]:
BRANCH = 'round-robin-v2'
REPO   = 'https://github.com/adikeshn/pcb-router-world.git'

import os, sys, dataclasses

BASE = '/content' if os.path.isdir('/content') else os.getcwd()
if os.path.basename(BASE) == 'pcb-router-world':
    BASE = os.path.dirname(BASE)
REPO_DIR = os.path.join(BASE, 'pcb-router-world')
os.chdir(BASE)

if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH --single-branch $REPO
os.chdir(REPO_DIR)

!git fetch origin $BRANCH -q
!git checkout -q $BRANCH
!git reset --hard -q origin/$BRANCH
!git log -1 --format='repo now at: %h %s'

!pip -q install -r requirements.txt
sys.path.insert(0, os.getcwd())

for m in [m for m in list(sys.modules) if m.startswith('pcb_router_rr2')]:
    del sys.modules[m]

from pcb_router_rr2.config import Config
fields = {f.name for f in dataclasses.fields(Config)}
missing = sorted({'budget_mm', 'n_dirs', 'turn_free_units', 'self_lookback_mm',
                  'constriction_penalty_total', 'spacing_reward_coeff',
                  'lr_anneal', 'checkpoint_every_steps'} - fields)
assert not missing, (
    f'\n\nSTALE CODE: config.py is missing {missing}.\n'
    f"Push the updated pcb_router_rr2/*.py to the '{BRANCH}' branch, then re-run.\n"
    f"Loaded from: {sys.modules['pcb_router_rr2.config'].__file__}")

print(f'setup complete — config.py is current ({len(fields)} parameters)')


## 2 · Weights & Biases


In [ ]:
import wandb
wandb.login()


## 3 · Hyperparameters

Maps 1:1 to `pcb_router_rr2.config.Config`. Unknown keys raise immediately.

**Dense terms are per-episode TOTALS**, divided internally by the round/step count,
so the balance survives changes to `budget_mm` or `step_mm`.


In [ ]:
HPARAMS = dict(
    # ---------------- board (mm) ----------------
    board_width_mm   = 200.0,
    board_height_mm  = 150.0,
    edge_clearance_mm= 2.0,
    connector_rect   = (78.0, 38.0, 122.0, 50.0),
    pins = [
        (83.5, 50.0), (89.0, 50.0), (94.5, 50.0), (100.0, 50.0),
        (105.5, 50.0), (111.0, 50.0), (116.5, 50.0),   # 7 on top edge
        (85.0, 38.0), (100.0, 38.0), (115.0, 38.0),    # 3 on bottom edge
    ],
    obstacles = [],

    # ---------------- clearances ----------------
    trace_clearance_mm    = 1.33,
    self_clearance_mm     = 0.60,   # MUST be < step_mm (validated)
    obstacle_clearance_mm = 1.00,

    # ---------------- growth ----------------
    step_mm    = 2.0,      # 50 rounds x 10 traces = 500 steps/episode
    budget_mm  = 100.0,    # SINGLE fixed budget
    n_dirs     = 16,       # 22.5 deg apart
    ban_reverse= True,
    use_breakout = False,

    # ---------------- dense reward (per-episode TOTALS) ----------------
    spacing_dense_total        = 0.80,
    dense_spacing_target_mm    = 16.0,
    constriction_penalty_total = 0.40,   # traces walling each other in
    constriction_comfort       = 4,
    path_penalty_total         = 0.25,
    path_soft_mm               = 4.0,
    self_penalty_total         = 0.15,   # weak: meandering is DESIRED
    self_soft_mm               = 3.0,    # = min acceptable meander width
    self_lookback_mm           = 12.0,   # must be >> self_soft_mm
    turn_penalty_total         = 0.30,
    turn_free_units            = 1,      # adjacent alternation is free
    edge_penalty_total         = 0.0,    # removed

    # ---------------- terminal reward ----------------
    w_terminal_base        = 12.0,   # risk dial: raise if gate_pass < 40%
    spacing_reward_coeff   = 1.50,   # x sqrt(mm), UNCAPPED but concave
    w_path_clearance_bonus = 1.50,
    terminal_clearance_target_mm = 14.0,
    w_endpoint_edge_bonus  = 0.0,    # removed
    endpoint_spec_mm       = 13.0,   # label + ranking tier only

    # ---------------- portfolio ----------------
    portfolio_k        = 5,
    min_moved_frac     = 0.6,
    min_point_shift_mm = 20.0,

    # ---------------- PPO ----------------
    total_timesteps = 8_000_000,
    n_envs        = 8,
    learning_rate = 3e-4,
    lr_anneal     = True,     # linear decay to zero
    n_steps       = 512,
    batch_size    = 512,
    n_epochs      = 6,
    gamma         = 0.999,    # horizon 1000 vs 500-step episodes
    ent_coef      = 0.01,
    net_arch      = [256, 256],
    device        = 'auto',

    # ---------------- eval / logging / checkpointing ----------------
    eval_every_steps        = 100_000,
    eval_episodes           = 5,
    checkpoint_every_steps  = 250_000,
    early_stop_patience_evals = 0,   # >0 to stop on sustained decline
    explorer_every_episodes = 200,
    explorer_episodes       = 5,
    render_every_episodes   = 250,
    log_every_episodes      = 10,

    wandb_project  = 'pcb-routing',
    wandb_run_name = None,
    wandb_mode     = 'online',
)

cfg = Config().override(**HPARAMS)
cfg.validate()

import torch
dev = 'cuda (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'cpu'
print(f'config OK — {cfg.n_traces} traces, {cfg.budget_rounds} rounds, {cfg.episode_steps} steps/episode')
print(f'gamma {cfg.gamma} -> horizon {1/(1-cfg.gamma):.0f} '
      f'({1/(1-cfg.gamma)/cfg.episode_steps:.1f}x episode)')
print(f'PPO updates on: {dev}')


## 4 · Board preview — check the layout BEFORE training


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from pcb_router_rr2.board import Board
from pcb_router_rr2.rendering import preview_figure

print(Board.from_config(cfg).summary())
print()
fig = preview_figure(cfg, save_path='board_preview.png')
plt.show()


## 5 · Acceptance checks

Each encodes a bug that actually shipped in an earlier version:

* **self_crossing_check** — a legal 3-step fold that crosses itself must be caught,
  while straight runs and 90° corners stay legal.
* **self_penalty_check** — the coiling penalty must be *silent* on straight runs and
  wide meanders, and *fire* on tight coils. An earlier version fired on every step.
* **turn_penalty_check** — alternating between adjacent directions is how a discrete
  grid approximates an intermediate heading; it must be free.
* **zero_violation_check** — zero audited violations. Note `frac_survived` rather than
  completion: at full budget the random policy completes ~0%, so completion saturates.
* **reward_scale_check** — discount-aware.


In [ ]:
from pcb_router_rr2.validate import run_all
assert run_all(cfg), 'Acceptance checks FAILED — do not train with this config.'


## 6 · Train

**Watch first:** `eval/gate_pass_rate` and `eval/evals_since_best`. A previous run
peaked at 91% gate-pass then collapsed to ~2% over 3.5M steps. `model_best.zip` is
now saved whenever eval gate-pass improves, and checkpoints land every 250k steps,
so the best policy can never be lost again.

Other key panels:
* `train/frac_survived` — progress even when nothing completes
* `train/most_boxed_trace` — which trace is dying, i.e. who is being walled in
* `train/min_freedom` — how close traces come to having no legal move
* `train/r_spacing` (uncapped, should climb) vs `train/q_clear` (capped, watch for 1.0)
* `reward/*` — per-term decomposition; a term at ~0% of budget is inert

Every board image is stamped with the episode and step count that produced it.


In [ ]:
from pcb_router_rr2.train import train
run_dir = train(cfg)
print('artifacts in', run_dir)


## 7 · Results


In [ ]:
import json
from IPython.display import Image, display

with open(f'{run_dir}/portfolio/portfolio.json') as f:
    index = json.load(f)

print(f'{len(index)} of {cfg.portfolio_k} portfolio slots filled')
for e in index:
    print(f"  rank {e['rank']} | terminal {e['reward_terminal']:.2f} | "
          f"spacing {e['min_endpoint_spacing_mm']:.1f}mm | "
          f"spec {'PASS' if e['meets_spec'] else 'miss'} | "
          f"self-gap {e['min_self_distance_mm']:.2f}mm | "
          f"turn rate {e['turn_rate']:.2f} | "
          f"found at episode {e['found_at_episode']}")
    display(Image(e['png'], width=560))


### Inference — generate boards from a trained model
```python
from pcb_router_rr2.inference import solve
portfolio = solve(cfg, f'{run_dir}/model_best.zip', n_episodes=200)
```

### Download everything
```python
!zip -r results.zip {run_dir}
from google.colab import files; files.download('results.zip')
```
